# PARC2026 π0.5 Fixed Simulator Eval — Fresh L4/A100 Runtime

新しい Colab GPU runtime から **Runtime → Run all** で最初から実行する notebook です。

- Google Drive に保存済みの 4 つの 150-step LoRA adapter を使用
- pinned LIBERO-plus / LIBERO / MuJoCo / robosuite を構築
- Colab の `matplotlib_inline` 問題を避けて常に `MPLBACKEND=Agg`
- 1 variantずつ `LoRA merge → Track1固定評価 → Drive保存 → merged checkpoint削除`
- 完了済み variant は再実行時に skip（resume可能）
- 学習 dataset は再ダウンロードしません

## Runtime
**L4 24GB 推奨**。最終採点の L4 24GB に近い推論条件です。A100でも可。T4は対象外です。

## 事前準備
Colab Secrets に `HF_TOKEN` を登録し、`google/paligemma-3b-pt-224` の利用条件に同意しておいてください。


In [ ]:
# 0/5 Fresh-runtime preflight + Drive
import os, json, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

print("=== 0/5 FRESH RUNTIME PREFLIGHT ===", flush=True)

gpu = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader,nounits"], text=True).strip()
print("GPU:", gpu)
if "T4" in gpu.upper():
    raise RuntimeError("T4 16GB は対象外。L4 または A100 に変更してください。")

free = shutil.disk_usage('/content').free/1024**3
print(f"local free: {free:.1f} GiB")
if free < 45:
    raise RuntimeError("fresh runtime ではありません。ローカル空き45GiB以上の新しいruntimeで再実行してください。")

drive.mount('/content/drive')
DRIVE_ABLATION = Path('/content/drive/MyDrive/parc2026-cache/pi05-ablation-group-aware-v2')
DRIVE_EVAL = Path('/content/drive/MyDrive/parc2026-cache/pi05-fixed-sim-eval-v2')
VARIANTS = [
    ('V0_RAW','colab_data_v0_raw'),
    ('V1_MULTI','colab_data_v1_multi'),
    ('V1_ALL_REVIEW','colab_data_v1_all_review'),
    ('V2_SQRT','colab_data_v2_sqrt'),
]
for v,r in VARIANTS:
    a=DRIVE_ABLATION/r/'pretrained_model'; s=DRIVE_ABLATION/r/'cheap_ablation_summary.json'
    assert a.is_dir(), a; assert s.is_file(), s
    print(f"{v:16s} adapter=FOUND summary=FOUND")
DRIVE_EVAL.mkdir(parents=True, exist_ok=True)

tok = userdata.get('HF_TOKEN')
if not tok:
    raise RuntimeError("Colab Secrets に HF_TOKEN を登録してください。")
os.environ['HF_TOKEN']=tok
print('HF_TOKEN: FOUND (hidden)')
print('=== PREFLIGHT: PASS ===')


In [ ]:
# 1/5 Clone pinned repo + Python 3.10 + π0.5 env
import os, shutil, subprocess, threading, time
from pathlib import Path
from collections import deque

ROOT=Path('/content/parc2026'); REPO=ROOT/'py_AI'; ROOT.mkdir(parents=True, exist_ok=True)
REF='1ff29d7ce685688df274f2db5136dcc6a9aa8132'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--all','--tags'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--force',REF],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==REF
print('repo HEAD:', REF)

if shutil.which('uv') is None:
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
uv=shutil.which('uv') or '/root/.local/bin/uv'
subprocess.run([uv,'python','install','3.10'],check=True)
PY310=subprocess.check_output([uv,'python','find','3.10'],text=True).strip(); print('python3.10:',PY310)

DATA_ROOT=ROOT/'cache/pi05-eval'; LEROBOT_ROOT=ROOT/'vendor/lerobot-pi05-eval'; VENV=LEROBOT_ROOT/'.venv'
PY=VENV/'bin/python'; PIP=VENV/'bin/pip'

def gpu_status():
    try:
        return [x.strip() for x in subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader,nounits'],text=True).strip().split(',')]
    except: return ['?','?','?']
def free_gib(): return shutil.disk_usage('/content').free/1024**3

def run_live(label,cmd,cwd=None,env=None,log=None,hb=20):
    print('\n'+'='*76+'\n'+label+'\n'+'='*76,flush=True)
    fp=open(log,'w',buffering=1) if log else None
    p=subprocess.Popen(cmd,cwd=str(cwd) if cwd else None,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    tail=deque(maxlen=120)
    def rd():
        for ln in iter(p.stdout.readline,''):
            ln=ln.rstrip(); tail.append(ln); print('[child]',ln,flush=True)
            if fp: fp.write(ln+'\n')
    t=threading.Thread(target=rd,daemon=True); t.start(); st=time.time()
    while p.poll() is None:
        u,m,mt=gpu_status(); e=int(time.time()-st)
        print(f'[heartbeat] {label} | {e//60:02d}:{e%60:02d} | GPU={u}% | VRAM={m}/{mt} MiB | free={free_gib():.1f} GiB',flush=True)
        time.sleep(hb)
    t.join(10)
    if fp: fp.close()
    print('[finished] rc=',p.returncode,flush=True)
    if p.returncode:
        print('\n'.join(tail)); raise RuntimeError(f'{label} failed rc={p.returncode}')

setup=REPO/'examples/pi05_libero_finetune/scripts/setup_train.sh'
env=os.environ.copy(); env.update({'PYTHONUNBUFFERED':'1','PYTHON':PY310,'DATA_ROOT':str(DATA_ROOT),'LEROBOT_ROOT':str(LEROBOT_ROOT),'INSTALL_FFMPEG':'1','HF_TOKEN':os.environ['HF_TOKEN']})
run_live('1/5 SETUP PI0.5 ENV',['bash',str(setup)],cwd=REPO/'examples/pi05_libero_finetune',env=env,log='/content/drive/MyDrive/parc2026-cache/pi05-fixed-sim-eval-v2/00_setup_pi05_env.log')
assert PY.exists(); assert PIP.exists()
run_live('1/5 PI0.5 IMPORT GATE',[str(PY),'-u','-c',"import torch,lerobot; from lerobot.policies.pi05.configuration_pi05 import PI05Config; print(torch.__version__, torch.cuda.is_available(), lerobot.__version__, PI05Config().drop_n_last_frames)"],env=env)
print('=== PI0.5 ENV: PASS ===')


In [ ]:
# 2/5 Build pinned simulator + headless import gate
import os, subprocess, shutil
from pathlib import Path
ROOT=Path('/content/parc2026'); REPO=ROOT/'py_AI'; SIM=ROOT/'sim'; LP=SIM/'LIBERO-plus'; L=SIM/'LIBERO'
LEROBOT_ROOT=ROOT/'vendor/lerobot-pi05-eval'; PY=LEROBOT_ROOT/'.venv/bin/python'; PIP=LEROBOT_ROOT/'.venv/bin/pip'
DRIVE_EVAL=Path('/content/drive/MyDrive/parc2026-cache/pi05-fixed-sim-eval-v2')

script=f"""
set -euo pipefail
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends libmagickwand-dev libosmesa6 libosmesa6-dev libgl1 libglfw3 libglew-dev libegl1 libsm6 libxext6 libxrender-dev libglib2.0-0 unzip
"{PIP}" install -q --timeout 180 --retries 5 "numpy==1.26.4" "mujoco==3.7.0" "robosuite==1.4.0" "gym==0.25.2" "bddl==3.6.0" "cloudpickle==3.1.2" "easydict==1.13" "hydra-core==1.3.2" "einops==0.8.2" "opencv-python-headless==4.11.0.86" scipy pyyaml h5py Pillow termcolor tqdm matplotlib requests msgpack fastapi uvicorn huggingface_hub wand scikit-image
mkdir -p "{SIM}"
clone_pinned() {{ d="$1"; u="$2"; r="$3"; if [ ! -d "$d/.git" ]; then git init -q "$d"; git -C "$d" remote add origin "$u"; git -C "$d" fetch -q --depth 1 origin "$r"; git -C "$d" checkout -q FETCH_HEAD; fi; }}
clone_pinned "{LP}" https://github.com/sylvestf/LIBERO-plus 4976dc30028e805ff8094b55501d532c48fec182
clone_pinned "{L}" https://github.com/Lifelong-Robot-Learning/LIBERO 8f1084e3132a39270c3a13ebe37270a43ece2a01
touch "{LP}/libero/__init__.py" "{LP}/libero/libero/__init__.py"
sed -i 's/torch.load(init_states_path)/torch.load(init_states_path, weights_only=False)/' "{LP}/libero/libero/benchmark/__init__.py" || true
ASSETS="{LP}/libero/libero/assets"; COUNT=0; [ -d "$ASSETS/textures" ] && COUNT=$(find "$ASSETS/textures" -maxdepth 1 -type f | wc -l)
if [ "$COUNT" -lt 100 ]; then
 rm -rf "{SIM}/.tmp_assets"; mkdir -p "{SIM}/.tmp_assets"
 HF_TOKEN="${{HF_TOKEN}}" "{PY}" - <<'PYASSET'
from huggingface_hub import hf_hub_download
hf_hub_download('Sylvest/LIBERO-plus','assets.zip',repo_type='dataset',revision='dd2bd61b7d9a6fef1abc52d606e983b41886a149',local_dir='{SIM}/.tmp_assets')
PYASSET
 unzip -q "{SIM}/.tmp_assets/assets.zip" -d "{LP}/libero/libero"; rm -rf "{SIM}/.tmp_assets"
 COUNT=0; [ -d "$ASSETS/textures" ] && COUNT=$(find "$ASSETS/textures" -maxdepth 1 -type f | wc -l)
 if [ "$COUNT" -lt 100 ]; then N=$(find "{LP}/libero/libero" -type d -path '*/assets' | grep -v '^{LP}/libero/libero/assets$' | head -1); [ -n "$N" ] && rm -rf "$ASSETS" && ln -sfn "$(realpath "$N")" "$ASSETS"; fi
fi
test -e "{LP}/libero/libero/assets/scenes/libero_floor_base_style.xml"
ln -sfn "{REPO}/compe/t3/assets/bddl_files/libero_t3" "{LP}/libero/libero/bddl_files/libero_t3"
ln -sfn "{REPO}/compe/t3/assets/init_files/libero_t3" "{LP}/libero/libero/init_files/libero_t3"
mkdir -p "$HOME/.libero"
cat > "$HOME/.libero/config.yaml" <<EOF
benchmark_root: {LP}/libero/libero
bddl_files: {LP}/libero/libero/bddl_files
init_states: {LP}/libero/libero/init_files
datasets: {LP}/libero/libero/datasets
assets: {L}/libero/libero/assets
EOF
rm -rf /var/lib/apt/lists/*; "{PIP}" cache purge >/dev/null 2>&1 || true
"""
env=os.environ.copy(); env.update({'PYTHONUNBUFFERED':'1','HF_TOKEN':os.environ['HF_TOKEN'],'MPLBACKEND':'Agg','MUJOCO_GL':'egl'})
run_live('2/5 SIMULATOR SETUP',['bash','-lc',script],cwd=ROOT,env=env,log=DRIVE_EVAL/'01_sim_setup.log')
sim_env=os.environ.copy(); sim_env.update({'PYTHONUNBUFFERED':'1','HF_TOKEN':os.environ['HF_TOKEN'],'MPLBACKEND':'Agg','MUJOCO_GL':'egl','PYTHONPATH':f'{LP}:{REPO}:{REPO/"compe"}'})
gate="""import os; print('MPLBACKEND=',os.environ.get('MPLBACKEND')); import numpy,mujoco,robosuite; import libero.libero.benchmark; from compe.t1 import register_t1; from compe.t2 import register_t2; from compe.t3 import register_t3; register_t1(); register_t2(); register_t3(); print('numpy',numpy.__version__,'mujoco',mujoco.__version__,'robosuite',robosuite.__version__); print('=== LIBERO SUITES: PASS ===')"""
run_live('2/5 SIMULATOR IMPORT GATE',[str(PY),'-u','-c',gate],cwd=REPO,env=sim_env,log=DRIVE_EVAL/'02_import_gate.log')
free=shutil.disk_usage('/content').free/1024**3; print(f'free after setup: {free:.1f} GiB')
if free<20: raise RuntimeError('Simulator後の空き20GiB未満。fresh runtimeでやり直してください。')
print('=== SIMULATOR: PASS ===')


In [ ]:
# 3/5 Unattended fixed eval (4 variants, resume-safe)
import os,json,time,shutil,signal,threading,subprocess,urllib.request
from pathlib import Path
from collections import deque
ROOT=Path('/content/parc2026'); REPO=ROOT/'py_AI'; LP=ROOT/'sim/LIBERO-plus'; LEROBOT_ROOT=ROOT/'vendor/lerobot-pi05-eval'; PY=LEROBOT_ROOT/'.venv/bin/python'
DRIVE_ABLATION=Path('/content/drive/MyDrive/parc2026-cache/pi05-ablation-group-aware-v2'); DRIVE_EVAL=Path('/content/drive/MyDrive/parc2026-cache/pi05-fixed-sim-eval-v2')
LOCAL=ROOT/'outputs/pi05-fixed-sim-eval-v2'; MERGED=ROOT/'tmp/pi05-merged-eval-v2'; LOCAL.mkdir(parents=True,exist_ok=True); MERGED.mkdir(parents=True,exist_ok=True)
POLICY=REPO/'examples/pi05_libero_finetune/submission/policy_server.py'; MERGE=REPO/'examples/pi05_libero_finetune/scripts/merge_lora.py'
VARIANTS=[('V0_RAW','colab_data_v0_raw'),('V1_MULTI','colab_data_v1_multi'),('V1_ALL_REVIEW','colab_data_v1_all_review'),('V2_SQRT','colab_data_v2_sqrt')]
N_EPISODES=8; MAX_TASKS=4; MAX_STEPS=300; SEED=20260905; PORT=8000; overall=time.time()
def elapsed(st):
 e=int(time.time()-st); return f'{e//3600:02d}:{(e%3600)//60:02d}:{e%60:02d}'
def free_gib(): return shutil.disk_usage('/content').free/1024**3
def gpu():
 try: return [x.strip() for x in subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader,nounits'],text=True).strip().split(',')]
 except: return ['?','?','?']
def stream(stage,cmd,cwd=None,env=None,log=None):
 print('\n'+'='*76+'\n'+stage+'\n'+'='*76,flush=True); fp=open(log,'w',buffering=1) if log else None; tail=deque(maxlen=150)
 p=subprocess.Popen(cmd,cwd=str(cwd) if cwd else None,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
 def rd():
  for ln in iter(p.stdout.readline,''):
   ln=ln.rstrip(); tail.append(ln); print('[child]',ln,flush=True); fp.write(ln+'\n') if fp else None
 t=threading.Thread(target=rd,daemon=True); t.start(); st=time.time()
 while p.poll() is None:
  u,m,mt=gpu(); print(f'[heartbeat] {stage} | stage={elapsed(st)} | total={elapsed(overall)} | GPU={u}% | VRAM={m}/{mt} MiB | free={free_gib():.1f} GiB',flush=True); time.sleep(30)
 t.join(10); fp.close() if fp else None; print(f'[finished] rc={p.returncode} elapsed={elapsed(st)}')
 if p.returncode: print('\n'.join(tail)); raise RuntimeError(f'{stage} failed rc={p.returncode}')
def kill(p):
 if p and p.poll() is None:
  try: os.killpg(os.getpgid(p.pid),signal.SIGTERM); p.wait(15)
  except:
   try: os.killpg(os.getpgid(p.pid),signal.SIGKILL)
   except: pass
sim_env=os.environ.copy(); sim_env.update({'PYTHONUNBUFFERED':'1','HF_TOKEN':os.environ['HF_TOKEN'],'MPLBACKEND':'Agg','MUJOCO_GL':'egl','PYTHONPATH':f'{LP}:{REPO}:{REPO/"compe"}'})
print('=== 3/5 FIXED SIM EVAL ==='); print('protocol: track1 / 4 tasks / 8 episodes / seed',SEED); print('free:',f'{free_gib():.1f} GiB')
results=[]
for i,(v,r) in enumerate(VARIANTS,1):
 d=DRIVE_EVAL/v; done=d/'_DONE.json'
 if done.exists(): print('[resume]',v,'DONE -> skip'); results.append(json.loads(done.read_text())); continue
 if free_gib()<20: raise RuntimeError(f'{v}: free {free_gib():.1f} GiB <20 before merge')
 adapter=DRIVE_ABLATION/r/'pretrained_model'; merged=MERGED/v; shutil.rmtree(merged,ignore_errors=True)
 stream(f'3.{i}a MERGE {v}',[str(PY),'-u',str(MERGE),'--adapter',str(adapter),'--out',str(merged)],cwd=REPO,env=sim_env,log=DRIVE_EVAL/f'{v}_merge.log')
 server_log=open(DRIVE_EVAL/f'{v}_server.log','w',buffering=1); se=sim_env.copy(); se['PI05_CKPT']=str(merged)
 p=subprocess.Popen([str(PY),'-u',str(POLICY),'--port',str(PORT),'--device','cuda'],cwd=str(POLICY.parent),env=se,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True); tail=deque(maxlen=150)
 def rs():
  for ln in iter(p.stdout.readline,''): ln=ln.rstrip(); tail.append(ln); print(f'[server {v}]',ln,flush=True); server_log.write(ln+'\n')
 threading.Thread(target=rs,daemon=True).start(); st=time.time(); ready=False
 while time.time()-st<600:
  if p.poll() is not None: break
  try:
   with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health',timeout=3) as q:
    if q.status==200: ready=True; break
  except: pass
  u,m,mt=gpu(); print(f'[heartbeat] server-load {v} | {elapsed(st)} | GPU={u}% | VRAM={m}/{mt} MiB | free={free_gib():.1f} GiB'); time.sleep(20)
 if not ready: kill(p); server_log.close(); print('\n'.join(tail)); raise RuntimeError(v+' server failed')
 loc=LOCAL/v; shutil.rmtree(loc,ignore_errors=True); loc.mkdir(parents=True)
 try:
  stream(f'3.{i}b EVAL {v}',[str(PY),'-u','-m','pipeline','--server-url',f'http://127.0.0.1:{PORT}','--track','track1','--n-episodes',str(N_EPISODES),'--max-tasks',str(MAX_TASKS),'--max-steps',str(MAX_STEPS),'--seed',str(SEED),'--timeout','10','--output-dir',str(loc)],cwd=REPO,env=sim_env,log=DRIVE_EVAL/f'{v}_eval.log')
 finally: kill(p); server_log.close()
 d.mkdir(parents=True,exist_ok=True); dst=d/'pipeline_output'; shutil.rmtree(dst,ignore_errors=True); shutil.copytree(loc,dst)
 files=sorted(loc.glob('*.json'),key=lambda x:x.stat().st_mtime); score=None; tasks=[]
 if files:
  z=json.loads(files[-1].read_text()); tr=z.get('tracks',[])
  if tr: score=tr[0].get('overall_score'); tasks=tr[0].get('task_scores',[])
 status={'variant':v,'run_name':r,'track':'track1','max_tasks':MAX_TASKS,'episodes_per_task':N_EPISODES,'seed':SEED,'max_steps':MAX_STEPS,'overall_score':score,'task_scores':tasks,'repo_sha':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),'completed_at_unix':time.time()}
 done.write_text(json.dumps(status,indent=2)); results.append(status); print('SAVED',v,'score=',score)
 shutil.rmtree(merged,ignore_errors=True); time.sleep(5); print('free after cleanup:',f'{free_gib():.1f} GiB')
summary={'protocol':{'track':'track1','max_tasks':MAX_TASKS,'episodes_per_task':N_EPISODES,'seed':SEED,'max_steps':MAX_STEPS},'results':results}; (DRIVE_EVAL/'screening_summary.json').write_text(json.dumps(summary,indent=2))
rank=sorted([x for x in results if x.get('overall_score') is not None],key=lambda x:x['overall_score'],reverse=True)
print('\n=== FIXED SIM EVAL COMPLETE ===')
for i,x in enumerate(rank,1): print(f"{i}. {x['variant']:20s} score={x['overall_score']}")
print('Drive:',DRIVE_EVAL); print('total:',elapsed(overall)); print('=== UNATTENDED FIXED SIM EVAL: PASS ===')


In [ ]:
# 4/5 Read persisted results after reconnect
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/parc2026-cache/pi05-fixed-sim-eval-v2/screening_summary.json')
assert p.exists(), p
d=json.loads(p.read_text()); print('protocol:',d['protocol']); print()
rows=sorted([r for r in d['results'] if r.get('overall_score') is not None],key=lambda r:r['overall_score'],reverse=True)
for i,r in enumerate(rows,1): print(f"{i}. {r['variant']:20s} score={r['overall_score']}")
print('=== RESULTS READBACK: PASS ===')
